In [1]:
%cd ../

/nas/zhangtianning.di/projects/unique_data_build


/home/zhangtianning.di/anaconda3/envs/arxive/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from grobid_client.grobid_client import GrobidClient
client = GrobidClient(config_path="./config.json")


GROBID server is up and running


In [17]:
def save_linefile(path, data):
    with open(path,'w') as f:
        for line in data:
            f.write(line+'\n')

In [4]:
def read_linefile(path):
    if os.path.exists(path):
        with open(path,'r') as f:
            out = [t.strip() for t in f]
    else:
        out = []
    return out

In [ ]:
import subprocess
import sys 
import threading
import queue
import logging
def monitor_process(process, log_queue):
    # Monitor the stdout and stderr of the process
    for line in iter(process.stdout.readline, b''):
        decoded_line = line
        print(decoded_line, end='')  # Print the output in real-time
        log_queue.put(decoded_line)  # Add output to the queue for logging
        if "Error" in decoded_line:
            process.kill()
            break
    process.stdout.close()
from grobid_client.grobid_client import GrobidClient

client = GrobidClient(config_path="./config.json")



    


In [30]:
def process_batch(
    self,
    service,
    input_files,
    output=None,
    n=10,
    generateIDs=False,
    consolidate_header=True,
    consolidate_citations=False,
    include_raw_citations=False,
    include_raw_affiliations=False,
    tei_coordinates=False,
    segment_sentences=False,
    force=True,
    verbose=False,
    verbose=False,
):
    if verbose:
        print(len(input_files), "files to process in current batch")

    # we use ThreadPoolExecutor and not ProcessPoolExecutor because it is an I/O intensive process
    with concurrent.futures.ThreadPoolExecutor(max_workers=n) as executor:
        #with concurrent.futures.ProcessPoolExecutor(max_workers=n) as executor:
        results = []
        for input_file in input_files:
            selected_process = self.process_txt
            r = executor.submit(
                selected_process,
                service,
                input_file,
                generateIDs,
                consolidate_header,
                consolidate_citations,
                include_raw_citations,
                include_raw_affiliations,
                tei_coordinates,
                segment_sentences)

            results.append(r)

    for r in concurrent.futures.as_completed(results):
        input_file, status, text = r.result()
        filename = self._output_file_name(input_file, input_path, output)

        if status != 200 or text is None:
            print("Processing of", input_file, "failed with error", str(status), ",", text)
            # writing error file with suffixed error code
            try:
                pathlib.Path(os.path.dirname(filename)).mkdir(parents=True, exist_ok=True)
                with open(filename.replace(".grobid.tei.xml", "_"+str(status)+".txt"), 'w', encoding='utf8') as tei_file:
                    if text is not None:
                        tei_file.write(text)
                    else:
                        tei_file.write("")
            except OSError:
                print("Writing resulting TEI XML file", filename, "failed")
        else:
            # writing TEI file
            try:
                pathlib.Path(os.path.dirname(filename)).mkdir(parents=True, exist_ok=True)
                with open(filename,'w',encoding='utf8') as tei_file:
                    tei_file.write(text)
            except OSError:
               print("Writing resulting TEI XML file", filename, "failed")

In [31]:
client.process_batch("processCitationList", reference_path_list[:10],

                    )

TypeError: GrobidClient.process_batch() missing 11 required positional arguments: 'input_path', 'output', 'n', 'generateIDs', 'consolidate_header', 'consolidate_citations', 'include_raw_citations', 'include_raw_affiliations', 'tei_coordinates', 'segment_sentences', and 'force'

In [23]:
os.makedirs('/home/zhangtianning.di/tempfile')

In [25]:
!hostname

SH-IDCA1404-10-140-52-124


In [28]:
batchsize=1000
tempfile_txt = "/home/zhangtianning.di/tempfile/temp.txt"
tempfile_key = "/home/zhangtianning.di/tempfile/temp.key"
temp_part    = reference_string_part[:batchsize]
temp_key_list= [f"{a}<->{b}" for a,b,c in temp_part]
temp_txt_list= [c for a,b,c in temp_part]
save_linefile(tempfile_txt,temp_txt_list)
save_linefile(tempfile_key,temp_key_list)

In [7]:
reference_path_list = read_linefile("/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/all_ready_json.archive.reference.filelist")

In [29]:
reference_path_list[:1000]

['/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0001/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0018/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0006/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0020/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0023/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0027/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0017/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0025/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0029/uparxive/Reference',
 '/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_json/0704.0030/uparxive/Re

In [9]:
from tqdm.auto import tqdm

In [15]:
import os
reference_string_part = []
for i,reference_path in enumerate(tqdm(reference_path_list)):
    if i > 1000:break
    reference_path = reference_path.strip()
    arxivid = reference_path.split('/')[-3]
    reference_txt  = read_linefile(os.path.join(reference_path,'reference.txt'))
    reference_key  = read_linefile(os.path.join(reference_path,'reference.keys'))
    assert len(reference_txt) == len(reference_key)
    for key, txt in zip(reference_key,reference_txt):
        reference_string_part.append([arxivid, key, txt])

  0%|          | 0/1389734 [00:00<?, ?it/s]

In [ ]:
len(reference_string_part)

### The goal for this note is about correct parse the text value of citation 

For example,
- A. Y. Kitaev, quantum measurements and the Abelian stabilizer problem, arXiv eprint quant-ph/9511026, 1995. 
- I. L. Chuang, N. Gershenfeld, and M. Kubinec. Phys. Rev. Lett. 80, 3408 (1998) 

#### openalex data

In [46]:
openalex_json_list = list(Path("/home/zhangtianning.di/datasets/openalex/whole_paper_information/").glob("*/part*/metadata.jsonl"))
len(openalex_json_list)

497

In [70]:
str(p)

'/home/zhangtianning.di/datasets/openalex/whole_paper_information/updated_date=2023-09-30/part_000/metadata.jsonl'

In [54]:
metadatapath = "data/openalex/whole_paper_information/updated_date=2023-11-16/part_024/metadata.jsonl"

In [53]:
%cd ../

/home/zhangtianning.di/projects_local/unarXive/src


In [57]:
from tqdm.auto import tqdm

In [64]:
medatadata[90]

{'father_path': 'updated_date=2023-11-16/part_024.gz',
 'openalex_id': 'https://openalex.org/W1995337270',
 'doi': 'https://doi.org/10.1002/piuz.200690052',
 'title': 'Garen mit Wahrscheinlichkeit',
 'publication_date': '2006-05-01',
 'language': 'de',
 'authors': ['Thomas A. Vilgis'],
 'location': [{'journal': 'Physik in unserer Zeit',
   'issn_l': '0031-9252',
   'issn': ['0031-9252', '1521-3943'],
   'publisher': 'Wiley',
   'url': 'https://doi.org/10.1002/piuz.200690052',
   'key': 'piuz.200690052'}],
 'referenced_works': []}

In [58]:
medatadata= []
with open(metadatapath,'r') as f:
    for line in tqdm(f):
        medatadata.append(json.loads(line))

0it [00:00, ?it/s]

KeyboardInterrupt: 

#### detect arxiv label 

In [2]:
import os
import json
from tqdm.auto import tqdm, trange
from collections import defaultdict
from nameparser import HumanName
from pathlib import Path

In [4]:
ref_json_list = list(Path("/home/zhangtianning.di/datasets/whole_arxiv_data/whole_arxiv_quant_ph_json").glob("*/reference.txt"))
len(ref_json_list)

10188

In [31]:
import re
ARXIV_URL_PATT = re.compile(
    r'arxiv\.org\/[a-z0-9-]{1,10}\/(([a-z0-9-]{1,15}\/)?[\d\.]{4,9}\d)',
    re.I
)
ARXIV_ID_PATT = re.compile(
    r'arXiv:(([a-z0-9-]{1,15}\/)?[\d\.]{4,9}\d)',
    re.I
)
ARXIV_ID_PATT_DATE = re.compile(
    r'^([a-zA-Z-\.]+)?\/?(\d\d)(\d\d)(.*)$'
)
DOI_PATT = re.compile(
    # a DOI at the *end* of the string
    # NOTE: might match a "/" at the end
    r'10.\d{4,9}/[-._;()/:A-Z0-9]+$',
    re.I
)
FORMULA_PATT = re.compile(
    r'\{\{formula:.{36}\}\}',
    re.I
)
APS_DOI_PATT = re.compile(
    r'('  # either
    r'(rev\.\s*mod\.\s*phys\.?)'  # rev mod phys  -> g2
    '|'  # or
    r'(phys\.\s*rev\.\s*)'  # phys rev  -> g3
    r')'
    r'(([a-z]+\.? )+)[.,]?\s*'  # optional sequence of specifiers -> g4
    #                           # e.g.: <none>, a, b, lett,
    #                           #       accel beams, spec top phys ed res
    r'(\d+)\s*'  # issue number  -> g6
    r'(\([0-9a-z\s,.]+\))?[.,]?\s*'  # naughty year inbetween
    r'([a-z]?\d+)',  # paper identifier (can contain a leading letter)  -> g8
    re.I
)


def find_arxiv_id(text):
    """ Loor for an arXiv ID within the given text.
    """

    match = ARXIV_ID_PATT.search(text)
    if match:return match.group(1)
    else:
        match = ARXIV_URL_PATT.search(text)
        if match:return match.group(1)
        else:
            match=ARXIV_ID_PATT_DATE.search(text)
            if match:return match.group(1)
    return False


def identify_implicit_aps_journal_doi(bibstr):
    """ Identify DOI information in physics references to
        Journals of the American Physical Society.

        Example in/outputs:

        H. R. Riedl et al., Phys. Rev. 162, 692 (1967).
            -> 10.1103/physrev.162.692

        L. Davidovich et al., Phys. Rev. A 50, R895 (1994).
            -> 10.1103/physreva.50.R895

        Phys. Rev. B 88 (Jul, 2013) 045102
            -> 10.1103/physrevb.88.045102

        K. Zuza et al., Phys. Rev. Spec. Top. Phys. Ed. Res. 10, 010122 (2014).
            -> 10.1103/PhysRevSTPER.10.010122

        A. J. Leggett, Rev. Mod. Phys. 73, 307 (2001).
            -> 10.1103/revmodphys.73.307
    """

    m = APS_DOI_PATT.search(bibstr)
    if not m:
        # no match
        return None

    aps_doi_numid = '10.1103'
    rev_mod_phys = m.group(2)
    phys_rev = m.group(3)
    issue_number = m.group(6).lower()
    paper_id = m.group(8).lower()
    if rev_mod_phys is not None:
        journal_id = 'revmodphys'
    elif phys_rev is not None:
        journal_id = 'physrev'
    else:
        # something went wrong
        raise ValueError

    # post processing of phys rev appendix match
    phys_rev_spec = ''
    phys_rev_spec_raw = m.group(4)
    if phys_rev_spec_raw:
        phys_rev_spec = re.sub(
            '[\s.]+',
            '',
            phys_rev_spec_raw
        ).lower()
        if phys_rev_spec == 'spectopphysedres':
            phys_rev_spec = 'stper'  # special case
    # / post processing of phys rev appendix match

    doi = '{}/{}{}.{}.{}'.format(
        aps_doi_numid,
        journal_id,
        phys_rev_spec,
        issue_number,
        paper_id
    )
    return doi



In [42]:
def extract_arxiv_ids(text):
    pattern = r'\b(?:[a-z-A-Z-]+(?:\.[A-Z]{2})?)/\d{7}\b'
    return re.findall(pattern, text)

# Example usage:
citation_text = "A fascinating paper on string theory is found in Cond-Mat/0701491, while another interesting paper in mathematical physics is located at math-ph/9909024. For nonlinear dynamics, see nlin.CD/0108016."

arxiv_ids = extract_arxiv_ids(citation_text)
print(arxiv_ids)

['Cond-Mat/0701491', 'math-ph/9909024', 'nlin.CD/0108016']


In [40]:
extract_arxiv_ids("W. G. Unruh, UBC preprint, hep-th/9406058 (1994)")

[]

In [5]:
ref_json_list[0]

PosixPath('/home/zhangtianning.di/datasets/whole_arxiv_data/whole_arxiv_quant_ph_json/quant-ph_0407066/reference.txt')

In [7]:
whole_string_line = []
for p in tqdm(ref_json_list):
    with open(p, 'r') as lines:
        for line in lines:
            whole_string_line.append(line.strip())

  0%|          | 0/10188 [00:00<?, ?it/s]

In [21]:
find_arxiv_id(whole_string_line[60])

False

In [23]:
whole_string_line[60]

'P. Chen, and T. Tajima, 1999, Phys. Rev. Lett. , 83 , 256.'

In [22]:
identify_implicit_aps_journal_doi(whole_string_line[60])

'10.1103/physrevlett.83.256'

In [65]:
for t in whole_string_line:
    if 'quant-ph' not in t and ';' in t:
        a = find_arxiv_id(t)
        if a:
            #print(a)
            continue
        c = extract_arxiv_ids(t)
        if len(c)>0:
            #print(c)
            continue
#         b = identify_implicit_aps_journal_doi(t)
#         if b:
#             print(b)
#             continue
        print(f"===>{t}")

===>A. Furusawa et al. , Science 282 , 706 (1998); F. Grosshans et al. , Nature 421 , 238 (2003).
===>J. Jing et al. , Phys. Rev. Lett. 90 , 167903 (2003).;
===>H. Yonezawa, T. Aoki and A. Furusawa, Nature 431 , 430 (2004);
===>M. E. Smithers and E. Y. C. Lu, Phys. Rev. A 10 , 1874 (1974); R. R. Puri, Phys. Rev. A 50 , 5309 (1994); N. Piovella, M. Cola and R. Bonifacio, Phys. Rev. A 67 , 013817 (2003); S. Pirandola et al. , Phys. Rev. A 68 , 062317 (2003); A. Ferraro et al. , J. Opt. Soc. Am. B 21 , 1241 (2004); A. V. Rodionov and A. S. Chirkin, Pis’ma Zh. Éksp. Teor. Fiz. 79 , 311 (2004) [JETP Lett. 79 , 253 (2004)]; J. Guo et al. , Phys. Rev. A 71 , 034305 (2005).
===>S. L. Braunstein, C. A. Fuchs and H. J. Kimble, J. Mod. Opt. 47 , 267 (2000); K. Hammerer et al. , Phys. Rev. Lett. 94 , 150503 (2005).
===>J. Zakrzewski, D. Delande, and A. Buchleitner, 1995, Phys. Rev. Lett. , 75 , 1995; A. F. Brunello, T. Uzer, and D. Farrelly, 1996, Phys. Rev. Lett. , 76 , 4015; H. Maeda, and T. E. 

===>P.S. Jessen and I.H. Deutsch, Adv. At. Mol. Opt. Phys. 37 , 95 (1996); D.R. Meacher, Cont. Phys. 39 , 329 (1998); L. Guidoni and P. Verkerk, J. Opt. B 1 , R23 (1999).
===>C. I. Sukenik, M. G. Boshier, D. Cho, V. Sandoghdar, and E. A. Hinds, Phys. Rev. Lett. 70 , 5, 560 (1993); A. Anderson, S. Haroche, E. A. Hinds, W. Jhe, and D. Meschede, Phys. Rev. A 37 , 9, 3594 (1988).
===>F. Shimizu, Phys. Rev. Lett. 86 , 6, 987 (2001); F. Shimizu and J.-i. Fujita, ibid. 88 , 12, 123201 (2002); V. Druzhinina and M. DeKieviet, Phys. Rev. Lett. 91 , 193202 (2003).
===>V. Sandoghdar, C. I. Sukenik, E. A. Hinds, and S. Haroche, Phys. Rev. Lett. 68 , 23, 3432 (1992); M. Marrocco, M. Weidinger, R. T. Sang, and H. Walther, Phys. Rev. Lett. 81 , 26, 5784 (1998); M. A. Wilson, P. Bushev, J. Eschner, F. Schmidt-Kaler, C. Becher, R. Blatt, and U. Dorner, Phys, Rev. Lett. 91 , 21, 213602 (2003); P. Bushev, A. Wilson, J. Eschner, C. Raab, F. Schmidt-Kaler, C. Becher, and R. Blatt, ibid. 92 , 22, 223602 (200

===>C. Tsallis, J. Stat. Phys. 52 , 479 (1988); E.M.F. Curado and C. Tsallis, J. Phys. A 24 , L69 (1991) [Corrigenda: 24 , 3187 (1991) and 25 , 1019 (1992)]; C. Tsallis, R.S. Mendes and A.R. Plastino, Physica A 261 , 534 (1998). A regularly updated bibliography on the subject is accessible at http://tsallis.cat.cbpf.br/biblio.htm For reviews see C. Tsallis, in Nonextensive Statistical Mechanics and Thermodynamics , eds. S.R.A. Salinas and C. Tsallis, Braz. J. Phys. 29 , 1 (1999) [http://sbf.if.usp.br/ WWW - pages/Journals/BJP/Vol29/Num1/index.htm]; also, C. Tsallis, in Nonextensive Statistical Mechanics and Its Applications , eds. S. Abe and Y. Okamoto, Series Lecture Notes in Physics (Springer-Verlag, Berlin, 2001).
===>P.A. Alemany and D.H. Zanette, Phys. Rev. E 49 , R956 (1994); D.H. Zanette and P.A. Alemany, Phys. Rev. Lett. 75 , 366 (1995); C. Tsallis, S.V.F Levy, A.M.C. de Souza and R. Maynard, Phys. Rev. Lett. 75 , 3589 (1995) [Erratum: Phys. Rev. Lett. 77 , 5442 (1996)]; M. Bui

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)

